In [ ]:
from pathlib import Path
import json
from glob import glob

In [58]:
ROOT_FOLDER = Path("./GeneratedCorpus")
PT_NATIVO = ROOT_FOLDER / "PB NATIVO"
LEXICAL_SWAP = ROOT_FOLDER / "LEXICAL SWAP"

In [59]:
def extract_texts(file_path:Path) -> list:
    """
        There are two types of corpus:
        1. Starts with <tag> followed by [number of tokens]  separeted by '---'
        2. Multiple <tag> lines with no end of sentence token

        The function receives any of these types and extract the text
    """

    corpus = []
    
    curr_text = ""
    with open(file_path, "r") as file:
        while (line:= file.readline()) != '':
            
            if line.startswith("<") or line.startswith(" "):
                # Consume all '<' lines and add the new text to the corpus
                corpus.append(curr_text.strip())
                while (line:= file.readline()) == "<" or line.startswith(" "): pass
                curr_text = ""

            elif "---" in line:
                # Consume the token '---' and the white lines in between sentences
                while (line:= file.readline()) == " ":pass
                pass

            else:
                curr_text += line
        
        corpus.append(curr_text.strip())

    # Remove any white text that was left            
    corpus = list(filter(lambda x: len(x)>0, corpus))

    return corpus



In [60]:
VERBS = ["CAPACITAR", "COMPARTILHAR", "EXPLICAR", "EXPLORAR", "FAZER", "RECOMENDAR", "RECRIAR", "RELATAR"]

def get_language(file):
    """
        Look into the file name and return its respective language
    """
    if "PB NATIVO" in file: return "PB NATIVO"
    language = file.split(" ")[-1]
    language = language.replace(".txt", "")
    return language


for verb in VERBS:
    files = glob(f"**/*{verb}*.txt", recursive=True)

    corpora = dict()

    for file in files:
        corpora[get_language(file)] = extract_texts(file)

    json_filename = f"{verb}.json"

    with open(json_filename, "w") as file:
        json.dump(corpora, file, sort_keys=True, indent=4, ensure_ascii=False)


